Found XSRF token path from HAR file of a request to trains api: https://eticket.railway.uz/api/v3/handbook/trains/list


In [1]:
import json
import requests

with open("full_request.har", encoding="utf-8") as f:
    har = json.load(f)

for entry in har["log"]["entries"]:
    for header in entry["response"]["headers"]:
        if "xsrf" in header["value"].lower() or "xsrf" in header["name"].lower():
            print(entry["request"]["method"], entry["request"]["url"])
            print(f"  {header['name']}: {header['value']}")
            print()
            break

GET https://eticket.railway.uz/api/v1/csrf-token
  set-cookie: XSRF-TOKEN=b78ca210-a9c4-4c43-a9b9-bf95d520109b; Path=/



Testing token extraction.

In [2]:
session = requests.Session()
session.get("https://eticket.railway.uz/api/v1/csrf-token")
csrf_token = session.cookies.get("XSRF-TOKEN")
print(csrf_token)

2dce7182-bd5d-4286-b9b8-1a35938a3e77


Now make a test request to trains API with the token.

In [3]:
session = requests.Session()

# 1. Get CSRF token
session.get("https://eticket.railway.uz/api/v1/csrf-token")
csrf_token = session.cookies.get("XSRF-TOKEN")

# 2. Query trains
resp = session.post(
    "https://eticket.railway.uz/api/v3/handbook/trains/list",
    headers={
        "X-XSRF-TOKEN": csrf_token,
        "Content-Type": "application/json",
        "Accept": "application/json",
        "Accept-Language": "ru",
        "Referer": "https://eticket.railway.uz/ru/home",
        "Origin": "https://eticket.railway.uz",
        "device-type": "BROWSER",
    },
    json={
        "directions": {
            "forward": {
                "date": "2026-04-29",
                "depStationCode": "2900000",
                "arvStationCode": "2900700",
            }
        }
    },
)

print(resp.status_code)
print(json.dumps(resp.json(), indent=2, ensure_ascii=False))

200
{
  "data": {
    "directions": {
      "forward": {
        "trains": [
          {
            "type": "Высокоскоростной",
            "number": "768Ф",
            "departureDate": "29.04.2026 08:00",
            "timeOnWay": "02:25",
            "originRoute": {
              "depStationName": "ТАШКЕНТ Ц",
              "arvStationName": "САМАРКАНД"
            },
            "arrivalDate": "29.04.2026 10:25",
            "brand": "Afrosiyob",
            "cars": [],
            "subRoute": {
              "depStationName": "ТАШКЕНТ Ц",
              "depStationCode": "2900001",
              "arvStationName": "САМАРКАНД",
              "arvStationCode": "2900700"
            },
            "trainId": "768Ф_2026-04-29",
            "comment": null
          },
          {
            "type": "СК",
            "number": "710Ф",
            "departureDate": "29.04.2026 08:37",
            "timeOnWay": "03:09",
            "originRoute": {
              "depStationName": "Ташкент 

The brand field is cleanest binning key. Across both responses:

Afrosiyob → type varies inconsistently: "Высокоскоростной", "скрст", "СКРСТ" — but brand is always "Afrosiyob"

Sharq → brand: "Sharq", type "СК"

Nasaf → brand: "Nasaf", type "СК"

Everything else → brand is "Пассажирский", "скорый", "Cкорый" (note the Latin C in one of them!)

Departure stations. "depStationName": 
"ТАШКЕНТ": "2900000",
"САМАРКАНД": "2900700"
"БУХАРА": "2900800"

In [4]:
def request_trains(departure="2900000", arrival="2900700", date="2026-04-29"):
    session = requests.Session()
    session.get("https://eticket.railway.uz/api/v1/csrf-token")
    csrf_token = session.cookies.get("XSRF-TOKEN")

    resp = session.post(
        "https://eticket.railway.uz/api/v3/handbook/trains/list",
        headers={
            "X-XSRF-TOKEN": csrf_token,
            "Content-Type": "application/json",
            "Accept": "application/json",
            "Accept-Language": "ru",
            "Referer": "https://eticket.railway.uz/ru/home",
            "Origin": "https://eticket.railway.uz",
            "device-type": "BROWSER",
        },
        json={
            "directions": {
                "forward": {
                    "date": date,
                    "depStationCode": departure,
                    "arvStationCode": arrival,
                }
            }
        },
    )

    trains = resp.json()["data"]["directions"]["forward"]["trains"]
    return trains

BINS = {
    "Afrosiyob": "high_speed",
    "Sharq": "fast",
    "Nasaf": "fast",
}

def bin_train(train):
    return BINS.get(train["brand"], "basic")

In [ ]:
# Interesting stations, brands, classes, etc. can be hardcoded:
stations = {"Tashkent": "2900000", "Samarkand": "2900700", "Bukhara": "2900800"}
brands = ["Afrosiyob", "Sharq", "Nasaf"] # There are more, I typically care only about Afrosiyob
classes = {
    "Afrosiyob": {"2Е": "economic", "1С": "business"},
    "Sharq": {"2В": "economic", "1С": "business", "1В": "vip"},
    "Nasaf": {"2В": "economic"},
} # class codes differ by brand, so we need to specify them separately.

# Filter out by brand and availability
trains = request_trains(departure=stations["Tashkent"], arrival=stations["Samarkand"], date="2026-05-20")
for t in trains:
    if t["brand"] in brands and len(t["cars"]) > 0:
        for car in t["cars"]:
            print(car)
            for tariff in car["tariffs"]:
                print(f"{t["brand"]} {t["number"]} has {tariff['classServiceType']} {classes[t["brand"]].get(tariff["classServiceType"], "unknown")} seats available at {tariff['tariff']} UZS")


{'type': 'Biznes', 'freeSeats': 2, 'tariffs': [{'classServiceType': '1С', 'freeSeats': 2, 'tariff': 455000}], 'seatDetail': None}
Afrosiyob 768Ф has 1С business seats available at 455000 UZS
{'type': 'Ekonom', 'freeSeats': 7, 'tariffs': [{'classServiceType': '2Е', 'freeSeats': 7, 'tariff': 311000}], 'seatDetail': None}
Afrosiyob 768Ф has 2Е economic seats available at 311000 UZS
{'type': 'Сидячий', 'freeSeats': 35, 'tariffs': [{'classServiceType': '1С', 'freeSeats': 7, 'tariff': 455000}, {'classServiceType': '2Е', 'freeSeats': 28, 'tariff': 311000}], 'seatDetail': {'undef': 35, 'lateralDn': 0, 'lateralUp': 0, 'freeComp': 0, 'down': 0, 'up': 0}}
Afrosiyob 766Ф has 1С business seats available at 455000 UZS
Afrosiyob 766Ф has 2Е economic seats available at 311000 UZS
{'type': 'Сидячий', 'freeSeats': 1, 'tariffs': [{'classServiceType': '2Е', 'freeSeats': 1, 'tariff': 311000}], 'seatDetail': {'undef': 1, 'lateralDn': 0, 'lateralUp': 0, 'freeComp': 0, 'down': 0, 'up': 0}}
Afrosiyob 770Ф has 

In [11]:
trains


[{'type': 'Высокоскоростной',
  'number': '768Ф',
  'departureDate': '20.05.2026 08:00',
  'timeOnWay': '02:25',
  'originRoute': {'depStationName': 'ТАШКЕНТ Ц',
   'arvStationName': 'САМАРКАНД'},
  'arrivalDate': '20.05.2026 10:25',
  'brand': 'Afrosiyob',
  'cars': [{'type': 'Biznes',
    'freeSeats': 2,
    'tariffs': [{'classServiceType': '1С', 'freeSeats': 2, 'tariff': 455000}],
    'seatDetail': None},
   {'type': 'Ekonom',
    'freeSeats': 7,
    'tariffs': [{'classServiceType': '2Е', 'freeSeats': 7, 'tariff': 311000}],
    'seatDetail': None}],
  'subRoute': {'depStationName': 'ТАШКЕНТ Ц',
   'depStationCode': '2900001',
   'arvStationName': 'САМАРКАНД',
   'arvStationCode': '2900700'},
  'trainId': '768Ф_2026-05-20',
  'comment': None},
 {'type': 'СК',
  'number': '127Ф',
  'departureDate': '20.05.2026 01:10',
  'timeOnWay': '04:15',
  'originRoute': {'depStationName': 'Андижан 1', 'arvStationName': 'Кунград'},
  'arrivalDate': '20.05.2026 05:25',
  'brand': 'Пассажирский',
  

In [12]:
stations = {"Tashkent": "2900000", "Samarkand": "2900700", "Bukhara": "2900800"}
brands = ["Afrosiyob", "Sharq", "Nasaf"] # There are more, I typically care only about Afrosiyob
classes = ["2Е", "1С"] # Economic and business class
classes = {"Afrosiyob": {"2Е": "economic", "1С": "business"}, "Sharq": {"2Е": "economic"}, "Nasaf": {"2Е": "economic"}}